# שיעור 22 — קידוד קטגוריות בלי סדר מלאכותי
יחידה 3 | 2026–2027 | סדר מול One-Hot.

ידע קודם: קטגוריאלי שמי/סדרי, X/y ו־fit/transform. התאמה שלנו לסיכום מפגש 9 בהשתלמות, לא מחברת מקור של המרצה.
[CSV](https://raw.githubusercontent.com/samiabo75/Data-Science-ML-Course/main/Datasets/penguins.csv) | [מקור Palmer Penguins](https://github.com/allisonhorst/palmerpenguins) | [קטלוג](https://github.com/samiabo75/Data-Science-ML-Course/blob/main/Datasets/README.md).
ייחוס: Horst, Hill & Gorman (2020), palmerpenguins, DOI 10.5281/zenodo.3960218; גרסת הקורס 344×7. פרטי שימוש במקור. שמרו עותק אישי.

## פתיחה — האם אי 2 רחוק מאי 0 פי שניים מאי 1?
למודל מבוסס מרחק נצטרך בהמשך ייצוג מספרי של מאפיינים. אם נקודד Biscoe=0,Dream=1,Torgersen=2, ניצור סדר ומרחקים שלא נובעים משמות האיים. אין להשתמש במספרים רק מפני שהמחשב דורש מספרים.

אם יש סדר טבעי, מיפוי סדרי עשוי להיות שימושי, אך אינו מוכיח מרווחים שווים. בחירת הייצוג תלויה גם במודל. לא מאמנים מודל היום.

## 1. קטגוריה סדורה — הרחבה קצרה מחומרי ההשתלמות
cut ב־Diamonds מסודרת לפי Fair,Good,Very Good,Premium,Ideal. נמחיש ברשומות מומצאות, בלי טעינת מאגר נוסף. מיפוי 0–4 משמר סדר, אך החיסור בין הרמות אינו מדידה של פער איכות. ב־KNN גם מרווחים כאלה משפיעים על מרחק ולכן צריך לנמק את הבחירה; One-Hot היא חלופה גם כאשר יש סדר.

In [ ]:
import pandas as pd
cut_demo_df = pd.DataFrame({"cut": ["Fair", "Good", "Very Good", "Premium", "Ideal"]})
cut_order = {"Fair": 0, "Good": 1, "Very Good": 2, "Premium": 3, "Ideal": 4}
cut_encoded_demo = cut_demo_df.copy()
cut_encoded_demo["cut_order"] = cut_encoded_demo["cut"].map(cut_order)
display(cut_encoded_demo)

map מחליפה כל ערך לפי מילון שנקבע מראש מן המשמעות המקצועית, לא לפי תוצאות Test. קטגוריה שאינה במילון תהפוך לחסר במיפוי זה; אין להמשיך בשקט. אין להניח של־island יש סדר דומה.

## 2. One-Hot — עמודה לכל קטגוריה ידועה
לאיים שנלמדו באימון ניצור עמודות לדוגמה island_Biscoe,island_Dream,island_Torgersen. שורה של Dream תהיה 0,1,0 בסדר הזה.

1 מציין שייכות לקטגוריה ו־0 אי־שייכות. מספר התצפיות נשאר; מספר העמודות עשוי לגדול. נשמור את כל הקטגוריות בלי drop בשיעור זה כדי לא להעדיף קטגוריית ייחוס במרחק.

הקידוד מסיר סדר שרירותי אבל אינו מודד מרחק גאוגרפי בין איים, ואינו מבטיח שזה מאפיין טוב או מתאים לשימוש העתידי.

## 3. דוגמה אמיתית — מקודדים island בלבד
זו הדגמת הכנת מאפיין, לא המלצה לבנות מסווג על אי בלבד. מין ואי קשורים להרכב הדגימה; יש לבדוק מה המידע הזמין בזמן חיזוי ולאיזו אוכלוסייה רוצים להכליל. תווית species אינה בתוך X.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
url = "https://raw.githubusercontent.com/samiabo75/Data-Science-ML-Course/main/Datasets/penguins.csv"
df = pd.read_csv(url)
model_df = df.dropna(subset=["species"]).copy()
X = model_df[["island"]]
y = model_df["species"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
print("חסרים במאפיין האימון:")
print(X_train.isna().sum())
print("לפני קידוד:", X_train.shape, X_test.shape)

ב־island אין חסר בקובץ זה. טיפול בחסר קטגוריאלי היה דורש החלטה נפרדת שנלמדת/נקבעת בצורה תקינה; לא ממלאים אותו בממוצע.

In [ ]:
encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
encoder.set_output(transform="pandas")
encoder.fit(X_train)
X_train_encoded = encoder.transform(X_train)
X_test_encoded = encoder.transform(X_test)
print("קטגוריות שנלמדו באימון:", encoder.categories_)
print("אחרי קידוד:", X_train_encoded.shape, X_test_encoded.shape)
display(X_train_encoded.head(3))
display(X_test_encoded.head(3))

fit לומדת אילו קטגוריות קיימות באימון ובאיזה סדר עמודות. transform משתמשת באותו מבנה לאימון ולבדיקה. sparse_output=False ו־set_output מבקשות פלט טבלאי פשוט לקריאה; אין צורך ללמוד מבנים דלילים היום.

לא מבצעים fit נוסף על Test, ולא יוצרים בנפרד טבלאות דמה שעלולות לקבל עמודות שונות. בקובץ הנוכחי שלושת האיים מיוצגים באימון; אין הבטחה שכך יקרה בכל מאגר.

## 4. קטגוריה חדשה — חייבים להבין את המדיניות
handle_unknown="ignore" מייצגת קטגוריה שלא נראתה באימון בשורת אפסים עבור המאפיין הזה, במקום להוסיף עמודה או לבצע fit מחדש. זו מדיניות מוגבלת: היא אינה לומדת מידע על הקטגוריה החדשה. אפשר גם להגדיר מערכת שמתריעה/דוחה קלט לא מוכר, לפי הצורך.

דוגמת הוראה מומצאת: האימון מכיר A ו־B; הבדיקה מכילה C.

In [ ]:
toy_train = pd.DataFrame({"group": ["A", "B", "A"]})
toy_test = pd.DataFrame({"group": ["C", "A"]})
toy_encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
toy_encoder.set_output(transform="pandas")
toy_encoder.fit(toy_train)
toy_test_encoded = toy_encoder.transform(toy_test)
display(toy_test_encoded)

C קיבלה אפסים בשתי העמודות; A קיבלה 1 בעמודתה. לא הוכחנו שהתוצאה מתאימה לכל מודל או ש־C 'זהה' לקטגוריה קיימת. חובה לתעד התמודדות עם קלט לא מוכר.

## תוצר לפרויקט — יומן קידוד
ציינו שדה וסוגו, האם יש סדר, משמעות המיפוי, fit מאימון, שמות עמודות, מדיניות חסרים/חדשים ומגבלות. אין לקודד אוטומטית כל עמודה, ואין להכניס את היעד למאפיינים.

הסיכום בהשתלמות מציג get_dummies על Embarked. כאן נבחר OneHotEncoder כדי ללמוד את המבנה מאימון בלבד וליישם אותו על בדיקה. זו התאמה שלנו, לא שינוי שקט במקור. אין צורך לשנן שתי דרכים היום.

## כרטיס יציאה
1. למה 0,1,2 אינם מתאימים אוטומטית לשמות איים?
2. מה מייצג 1 בעמודת One-Hot?
3. האם מספר התצפיות גדל?
4. מאילו נתונים לומדים עמודות קידוד?
5. מה משמעות שורת אפסים לקטגוריה חדשה במדיניות שנבחרה?

היעד הוא תווית נפרדת; KNN של scikit-learn יכול לקבל כאן תוויות טקסט בלי One-Hot של y. Penguins להוראה, לא מאגר גמר מאושר.

In [ ]:
print("מקור:", df.shape)
print("עמודות מקודדות:", X_train_encoded.columns)